In [1]:
import math
from pathlib import Path

import ffmpeg
from tqdm.auto import tqdm


In [2]:
# =========================
# User settings
# =========================

input_dir = Path(
    "/Volumes/fsmresfiles/Basic_Sciences/Phys/ContractorLab/Projects/Common Files/JJM_YZ_CaImaging/LinearTrackDataAlignedAcrossDays/m328/BehavCamConcactenated_328"
)

# Match only original source videos, not files already created by this notebook
input_pattern = "*concactenatedbehavCam*.mp4"

# Rotation
angle_deg = 29.9

# Crop bounding box applied AFTER rotation + padding
# Format: (x, y, width, height)
crop_box = (48, 374, 671, 48)

# Output settings
rotated_suffix = "_rotated_padded"
cropped_suffix = "_rotated_cropped_output"
rotated_ext = ".mp4"
cropped_ext = ".avi"

# Compression / quality
rotated_codec = "mjpeg"   # good for ImageJ/FIJI-style workflows
cropped_codec = "mjpeg"   # keep AVI + MJPEG for compatibility
jpeg_quality = 3          # lower is better quality in ffmpeg MJPEG

# Appearance of padded regions after rotation
pad_color = "white"       # e.g. "white", "black", "0xFFFFFF"

# Behavior
overwrite_existing = False
delete_rotated_after_crop = False

print(f"Input directory: {input_dir}")
print(f"Rotation angle: {angle_deg} deg")
print(f"Crop box (x, y, width, height): {crop_box}")


Input directory: /Volumes/fsmresfiles/Basic_Sciences/Phys/ContractorLab/Projects/Common Files/JJM_YZ_CaImaging/LinearTrackDataAlignedAcrossDays/m328/BehavCamConcactenated_328
Rotation angle: 29.9 deg
Crop box (x, y, width, height): (48, 374, 671, 48)


In [3]:
def get_video_info(video_path):
    probe = ffmpeg.probe(str(video_path))
    video_stream = next(stream for stream in probe["streams"] if stream["codec_type"] == "video")
    width = int(video_stream["width"])
    height = int(video_stream["height"])

    duration = video_stream.get("duration")
    if duration is None:
        duration = probe.get("format", {}).get("duration")
    duration = float(duration) if duration is not None else None

    codec_name = video_stream.get("codec_name", "unknown")
    return {
        "width": width,
        "height": height,
        "duration": duration,
        "codec_name": codec_name,
    }


def rotated_canvas_size(width, height, angle_deg):
    angle_rad = math.radians(angle_deg)
    new_width = math.ceil(abs(width * math.cos(angle_rad)) + abs(height * math.sin(angle_rad)))
    new_height = math.ceil(abs(width * math.sin(angle_rad)) + abs(height * math.cos(angle_rad)))
    return new_width, new_height


def make_output_paths(input_path, rotated_suffix, cropped_suffix, rotated_ext, cropped_ext):
    stem = input_path.stem
    rotated_path = input_path.with_name(f"{stem}{rotated_suffix}{rotated_ext}")
    cropped_path = input_path.with_name(f"{stem}{cropped_suffix}{cropped_ext}")
    return rotated_path, cropped_path


def iter_progress_lines(process):
    while True:
        line = process.stdout.readline()
        if not line:
            break
        yield line.decode("utf-8", errors="replace").strip()


def run_ffmpeg_with_progress(stream, duration=None, desc="Processing"):
    process = (
        stream
        .global_args("-progress", "pipe:1", "-nostats")
        .run_async(pipe_stdout=True, pipe_stderr=True, overwrite_output=True)
    )

    pbar = tqdm(total=duration if duration is not None else 100,
                desc=desc,
                unit="s" if duration is not None else "%",
                dynamic_ncols=True)

    last_time = 0.0
    for line in iter_progress_lines(process):
        if line.startswith("out_time_ms=") and duration is not None:
            out_time_ms = int(line.split("=", 1)[1])
            current_time = out_time_ms / 1_000_000
            increment = max(0, current_time - last_time)
            if increment > 0:
                pbar.update(increment)
                last_time = current_time
        elif line.startswith("progress=") and duration is None:
            # fallback if duration is unknown
            if line.endswith("end"):
                pbar.n = pbar.total
                pbar.refresh()

    stderr = process.stderr.read().decode("utf-8", errors="replace")
    return_code = process.wait()
    pbar.close()

    if return_code != 0:
        raise RuntimeError(stderr)


def rotate_video(input_path, output_path, angle_deg, codec="mjpeg", qscale=3, pad_color="white"):
    info = get_video_info(input_path)
    width, height, duration = info["width"], info["height"], info["duration"]
    new_width, new_height = rotated_canvas_size(width, height, angle_deg)

    pad_x = (new_width - width) // 2
    pad_y = (new_height - height) // 2

    stream = (
        ffmpeg
        .input(str(input_path))
        .filter("pad", new_width, new_height, pad_x, pad_y, color=pad_color)
        .filter("rotate", f"{angle_deg}*PI/180", fillcolor=pad_color)
        .output(
            str(output_path),
            vcodec=codec,
            **{"q:v": qscale},
            vsync="vfr"
        )
    )

    run_ffmpeg_with_progress(stream, duration=duration, desc=f"Rotate: {input_path.name}")
    return output_path


def crop_video(input_path, output_path, crop_box, codec="mjpeg", qscale=3):
    x, y, w, h = crop_box
    info = get_video_info(input_path)
    width, height, duration = info["width"], info["height"], info["duration"]

    if x < 0 or y < 0 or w <= 0 or h <= 0:
        raise ValueError(f"Invalid crop_box {crop_box}. Expected non-negative x/y and positive width/height.")
    if x + w > width or y + h > height:
        raise ValueError(
            f"crop_box {crop_box} exceeds frame size {(width, height)} for {input_path.name}"
        )

    stream = (
        ffmpeg
        .input(str(input_path))
        .filter("crop", w, h, x, y)
        .output(
            str(output_path),
            vcodec=codec,
            **{"q:v": qscale}
        )
    )

    run_ffmpeg_with_progress(stream, duration=duration, desc=f"Crop:   {input_path.name}")
    return output_path


In [4]:
# Discover candidate source videos
all_inputs = sorted(input_dir.glob(input_pattern))

# keep only original inputs, not files already generated
source_videos = [
    p for p in all_inputs
    if rotated_suffix not in p.stem and cropped_suffix not in p.stem
]

print(f"Found {len(source_videos)} source video(s).")
for p in source_videos[:10]:
    print(" -", p.name)
if len(source_videos) > 10:
    print(" ...")


Found 11 source video(s).
 - m328_04062025_17_43_00_concactenatedbehavCam00_behavCam20.mp4
 - m328_04072025_18_15_56_concactenatedbehavCam00_behavCam20.mp4
 - m328_04082025_13_34_08_concactenatedbehavCam00_behavCam20.mp4
 - m328_04082025_15_20_26_concactenatedbehavCam00_behavCam20.mp4
 - m328_04082025_17_35_12_concactenatedbehavCam00_behavCam20.mp4
 - m328_04092025_14_33_02_concactenatedbehavCam00_behavCam20.mp4
 - m328_04092025_16_12_31_concactenatedbehavCam00_behavCam20.mp4
 - m328_04102025_15_59_58_concactenatedbehavCam00_behavCam20.mp4
 - m328_04102025_17_23_44_concactenatedbehavCam00_behavCam20.mp4
 - m328_04112025_18_15_12_concactenatedbehavCam00_behavCam20.mp4
 ...


In [ ]:
# Optional: inspect the first source video and confirm the crop box is in bounds after rotation
if not source_videos:
    raise FileNotFoundError(f"No matching source videos found in {input_dir}")

first_info = get_video_info(source_videos[0])
rot_w, rot_h = rotated_canvas_size(first_info["width"], first_info["height"], angle_deg)

print("First source video:", source_videos[0].name)
print("Original size:", (first_info["width"], first_info["height"]))
print("Original codec:", first_info["codec_name"])
print("Rotated padded size:", (rot_w, rot_h))
print("Crop box:", crop_box)

x, y, w, h = crop_box
if x + w > rot_w or y + h > rot_h:
    raise ValueError(
        f"crop_box {crop_box} exceeds rotated padded size {(rot_w, rot_h)}. "
        "Adjust crop_box before batch processing."
    )


In [5]:
# Run batch processing
results = []

for input_path in source_videos:
    rotated_path, cropped_path = make_output_paths(
        input_path, rotated_suffix, cropped_suffix, rotated_ext, cropped_ext
    )

    row = {
        "input": str(input_path),
        "rotated": str(rotated_path),
        "cropped": str(cropped_path),
        "status": "pending",
    }

    try:
        if rotated_path.exists() and not overwrite_existing:
            print(f"Skipping existing rotated file: {rotated_path.name}")
        else:
            rotate_video(
                input_path,
                rotated_path,
                angle_deg=angle_deg,
                codec=rotated_codec,
                qscale=jpeg_quality,
                pad_color=pad_color,
            )

        if cropped_path.exists() and not overwrite_existing:
            print(f"Skipping existing cropped file: {cropped_path.name}")
        else:
            crop_video(
                rotated_path,
                cropped_path,
                crop_box=crop_box,
                codec=cropped_codec,
                qscale=jpeg_quality,
            )

        if delete_rotated_after_crop and rotated_path.exists():
            rotated_path.unlink()

        row["status"] = "done"

    except Exception as e:
        row["status"] = f"error: {e}"
        print(f"ERROR for {input_path.name}: {e}")

    results.append(row)

print("Finished batch.")


Skipping existing rotated file: m328_04062025_17_43_00_concactenatedbehavCam00_behavCam20_rotated_padded.mp4
Skipping existing cropped file: m328_04062025_17_43_00_concactenatedbehavCam00_behavCam20_rotated_cropped_output.avi


Rotate: m328_04072025_18_15_56_concactenatedbehavCam00_behavCam20.mp4:   0%|              | 0/342.65 [00:00<?,…

Crop:   m328_04072025_18_15_56_concactenatedbehavCam00_behavCam20_rotated_padded.mp4:   0%| | 0/342.65 [00:00<…

Rotate: m328_04082025_13_34_08_concactenatedbehavCam00_behavCam20.mp4:   0%|              | 0/342.85 [00:00<?,…

Crop:   m328_04082025_13_34_08_concactenatedbehavCam00_behavCam20_rotated_padded.mp4:   0%| | 0/342.85 [00:00<…

...

Rotate: m328_04112025_18_15_12_concactenatedbehavCam00_behavCam20.mp4:   0%|          | 0/342.783333 [00:00<?,…

Crop:   m328_04112025_18_15_12_concactenatedbehavCam00_behavCam20_rotated_padded.mp4:   0%| | 0/342.783333 [00…

Rotate: m328_04112025_19_20_55_concactenatedbehavCam00_behavCam20.mp4:   0%|          | 0/342.766667 [00:00<?,…

Crop:   m328_04112025_19_20_55_concactenatedbehavCam00_behavCam20_rotated_padded.mp4:   0%| | 0/342.766667 [00…

Finished batch.


In [ ]:
# Summary
import pandas as pd

results_df = pd.DataFrame(results)
results_df
